In [1]:
import scanpy as sc 
import pandas as pd 
import numpy as np 

import matplotlib.pyplot as plt 
import seaborn as sns 

from sklearn.metrics import adjusted_rand_score
from sklearn.decomposition import PCA

import scipy.sparse as sp 
import warnings

warnings.filterwarnings("ignore")

import os
import ctypes
import sys

# 1. 先设置 R_HOME
os.environ["R_HOME"] = "/home/pxy/miniconda3/envs/r40/lib/R"

# 2. 【核心黑科技】手动加载 R 的动态库
# 这步操作等同于在终端里设置 LD_LIBRARY_PATH，专门解决 VS Code 找不到库的问题
try:
    # 这是 R 的核心库路径
    libR_path = "/home/pxy/miniconda3/envs/r40/lib/R/lib/libR.so"
    # 强制加载进内存
    ctypes.CDLL(libR_path, mode=ctypes.RTLD_GLOBAL)
    print("✅ 成功强制加载 libR.so")
except OSError as e:
    print(f"❌ 加载失败: {e}")

# 3. 然后再导入其他包
sys.path.append("..") 

import spCLUE
import rpy2.robjects as robjects
print("R 环境路径:", robjects.r['R.home']()[0])

spCLUE.fix_seed(0)

# 定义DLPFC数据集的12个切片ID
slice_ids = [
    "151507", "151508", "151509", "151510",
    "151669", "151670", "151671", "151672",
    "151673", "151674", "151675", "151676"
]

# 用于存储每个切片的ARI结果
ari_results = []

# 数据路径（请根据实际情况确认路径是否正确）
data_dir = '/home/pxy/home/pxy/data/DLPFC/st/'

print(f"Start processing {len(slice_ids)} slices...")

for sample_name in slice_ids:
    print(f"\n{'='*20} Processing Sample: {sample_name} {'='*20}")
    
    # 1. 设置簇的数量 (根据DLPFC数据集的已知Ground Truth)
    # 151669-151672 通常只有5层，其他切片为7层
    if sample_name in ["151669", "151670", "151671", "151672"]:
        n_clusters = 5
    else:
        n_clusters = 7
    
    try:
        # 2. 加载数据
        # 使用 read_visium 加载数据，路径拼接逻辑参考原文件
        adata = sc.read_visium(data_dir + sample_name)
        adata.var_names_make_unique()
        
        # 加载元数据 (Ground Truth)
        meta = pd.read_csv(data_dir + sample_name + "/metadata.tsv", sep="\t")
        meta = meta.set_index("barcode")
        adata.obs["Region"] = meta.loc[adata.obs_names, "layer_guess_reordered"]
        
        # 3. 数据预处理与构图
        # 原文件 Cell 6 的逻辑
        adata = spCLUE.preprocess(adata)
        adata.obsm["X_pca"] = PCA(n_components=200, random_state=0).fit_transform(adata.X)
        
        g_spatial = spCLUE.prepare_graph(adata, "spatial", n_neighbors=6)
        g_expr = spCLUE.prepare_graph(adata, "expr", n_neighbors=8)
        graph_dict = {"spatial": g_spatial, "expr": g_expr}
        
        # 4. 模型初始化与训练
        # 原文件 Cell 8 的逻辑
        # 注意：这里将 n_clusters 参数改为动态变量，与当前切片保持一致
        spCLUE_model = spCLUE.spCLUE(adata.obsm["X_pca"], graph_dict, n_clusters)
        _, adata.obsm["spCLUE"], att_beta = spCLUE_model.train()
        
        # 5. 聚类
        # 原文件 Cell 10 的逻辑
        refinement = True
        cluster_method = "mclust"
        spCLUE.clustering(
            adata,
            n_clusters,
            key="spCLUE",
            refinement=refinement,
            cluster_methods=cluster_method,
        )
        
        # 6. 计算 ARI
        # 原文件 Cell 12 的逻辑
        # 过滤掉 Ground Truth 为 NA 的区域
        adata_valid = adata[adata.obs.Region.notna()]
        ARI = adjusted_rand_score(adata_valid.obs["Region"], adata_valid.obs["mclust_refined"])
        
        print(f"Sample {sample_name} ARI: {ARI:.8f}")
        ari_results.append(ARI)
        
    except Exception as e:
        print(f"Error processing sample {sample_name}: {e}")

# 7. 输出最终统计结果
print(f"\n{'='*20} Final Results {'='*20}")
if ari_results:
    mean_ari = np.mean(ari_results)
    median_ari = np.median(ari_results)
    print(f"ARI per slice: {[round(x, 5) for x in ari_results]}")
    print(f"Mean ARI: {mean_ari:.4f}")
    print(f"Median ARI: {median_ari:.4f}")
else:
    print("No ARI results collected.")

✅ 成功强制加载 libR.so
R 环境路径: /home/pxy/miniconda3/envs/r40/lib/R
Start processing 12 slices...

==================== Processing Sample: 151507 ====================
normalized data ---------------->
正在构建图: spatial, 使用度量: cosine ...
  -> 使用空间坐标 (euclidean)
  -> 计算最近邻 (NearestNeighbors)...
  -> 对称化与归一化...
spatial graph created successfully <----

正在构建图: expr, 使用度量: cosine ...
  -> 使用 PCA 表达特征
  -> 计算最近邻 (NearestNeighbors)...
  -> 对称化与归一化...
expr graph created successfully <----

Building gated consensus graph...
   Consensus graph weight threshold (top 20%): 0.1322
   Retained 22119/88714 edges (24.9%) after filtering
✅ Gated consensus graph built (alpha=0.85, k=20, weight_threshold=0.1322)
Training Start =========================>


  4%|▍         | 20/500 [00:01<00:21, 22.39it/s]

epoch 10: 0.06922224254808924
  Batch Loss: 13.7319, Cluster Loss: 2.5641, Rec Loss: 10.3435, Contrastive Loss: 8.2437, GraphGuided Loss: 0.0000,Delta: 0.0000, Beta: 1, Kappa: 0.1
epoch 20: 0.09418608649046398
  Batch Loss: 13.6990, Cluster Loss: 2.5553, Rec Loss: 10.3356, Contrastive Loss: 8.0819, GraphGuided Loss: 0.0000,Delta: 0.0000, Beta: 1, Kappa: 0.1


  8%|▊         | 41/500 [00:01<00:10, 43.72it/s]

epoch 30: 0.21875330688718841
  Batch Loss: 13.6538, Cluster Loss: 2.5274, Rec Loss: 10.3280, Contrastive Loss: 7.9839, GraphGuided Loss: 0.0000,Delta: 0.0000, Beta: 1, Kappa: 0.1
epoch 40: 0.3448091065196088
  Batch Loss: 13.5712, Cluster Loss: 2.4609, Rec Loss: 10.3209, Contrastive Loss: 7.8932, GraphGuided Loss: 0.0000,Delta: 0.0000, Beta: 1, Kappa: 0.1


...

   [Gating] Boundary: 30.6%, Valid anchors: 26.0%
epoch 80: 0.44116559821594664
  Batch Loss: 15.7092, Cluster Loss: 2.0705, Rec Loss: 11.9576, Contrastive Loss: 7.2463, GraphGuided Loss: 3.1884,Delta: 0.3000, Beta: 1, Kappa: 0.1
   [Gating] Boundary: 33.4%, Valid anchors: 26.2%


 19%|█▉        | 95/500 [00:02<00:14, 28.16it/s]

   [Gating] Boundary: 38.0%, Valid anchors: 25.7%
epoch 90: 0.38810168891055696
  Batch Loss: 15.9510, Cluster Loss: 2.0558, Rec Loss: 11.9580, Contrastive Loss: 7.2523, GraphGuided Loss: 3.0300,Delta: 0.4000, Beta: 1, Kappa: 0.1
   [Gating] Boundary: 41.7%, Valid anchors: 26.5%


 20%|█▉        | 99/500 [00:02<00:10, 36.67it/s]


   [Gating] Boundary: 41.8%, Valid anchors: 26.2%
epoch 100: 0.37686192642519745
  Batch Loss: 16.1480, Cluster Loss: 2.0059, Rec Loss: 11.9587, Contrastive Loss: 7.2262, GraphGuided Loss: 2.9216,Delta: 0.5000, Beta: 1, Kappa: 0.1
fitting ...
  |======================================================================| 100%
Sample 151676 ARI: 0.48491307

==================== Final Results ====================
ARI per slice: [0.50408, 0.45109, 0.57383, 0.52142, 0.4261, 0.34366, 0.84526, 0.76735, 0.4442, 0.49236, 0.53852, 0.48491]
Mean ARI: 0.5327
Median ARI: 0.4982
